# HPC Ensemble: 1000 Generators

Scale the generative canary ensemble from K=5 to **K=1000** to make adaptive
attacks computationally infeasible.

**Workflow:**
1. **HPC training (Cells 1-4)** — train 1000 generators on HPC. For multi-GPU
   parallel training, prefer `train_ensemble_parallel.py` which spawns one worker
   per GPU. The training cell below is a sequential fallback that works anywhere.
2. **Local inference (Cells 5-8)** — after downloading checkpoints, run
   evaluation locally on the RTX 3070 Ti.

**Baselines we compare against:**
- Paper (zebra fixed canary): AdvPatch=0.974, UPC=0.936, TCEGA=0.807, Natural=0.871
- Fixed cls17: 0.935, 0.877, 0.896, 0.900
- 5-generator ensemble: 0.935, 0.908, 0.905, 0.929

## Section 1: HPC Training

In [ ]:
import os
import sys
import time
import glob

# Ensure parent directory (FFF codebase) is importable
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

# =============================================
#  CONFIGURATION
# =============================================
N_GENERATORS = 1000
Z_DIM = 128
CANARY_CLS_ID = 22
EPOCHS = 50
BATCH_SIZE = 5
LR = 1e-3
WEIGHT = 2.0
SAVE_EVERY = 50  # only final checkpoint

# For job splitting across HPC jobs, override these:
#   Job 1: GEN_START=0,   GEN_END=250
#   Job 2: GEN_START=250, GEN_END=500
#   ...
GEN_START = 0
GEN_END = N_GENERATORS

CHECKPOINT_DIR = './ensemble_1000/checkpoints'
DATA_DIR = '../Data/traineval/VOC07_YOLOv8/train_120'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Config: {N_GENERATORS} generators, {EPOCHS} epochs, bs={BATCH_SIZE}, lr={LR}")
print(f"Slice: [{GEN_START}, {GEN_END})")
print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
import torch
from generative_canary import GenerativeCanary

# GenerativeCanary architecture (~693K params)
#   Input:  z = [noise_128 || bbox_4]  = R^132
#   FC:     132 -> 256 -> 512 -> 1024  (with Dropout 0.3)
#   Deconv: 16x8x8 -> 8x16x16 -> 3x32x32
#   Up:     bilinear to 80x80
#   Output: (3, 80, 80) pixels in [0, 1]

_probe = GenerativeCanary(z_dim=Z_DIM)
num_params = sum(p.numel() for p in _probe.parameters())
print(f"GenerativeCanary parameters: {num_params:,}")
print(f"Checkpoint size (fp32): ~{num_params * 4 / (1024*1024):.2f} MB per generator")
print(f"Total for {N_GENERATORS}: ~{num_params * 4 * N_GENERATORS / (1024**3):.2f} GB")
del _probe

In [ ]:
from types import SimpleNamespace
from YOLOv8_Combiner import (
    add_defensivepatch_into_tensor, make_yolo_train_label,
    Yolov8Dataset, Canary, freeze_seed,
)
from ObjectDetector.fjn_yolov8 import FJN_YOLOv8 as YOLOv8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# In-memory dataset: preload all tensors once, reuse across all generators
class InMemoryYolov8Dataset(Yolov8Dataset):
    def __init__(self, *dargs, **dkwargs):
        super().__init__(*dargs, **dkwargs)
        print(f"  Preloading {self.img_ls_len} image pairs into RAM...")
        # Use explicit parent class reference because super() doesn't
        # work inside list comprehensions (list comps have their own
        # scope and lose the __class__ cell).
        self._cache = [
            Yolov8Dataset.__getitem__(self, i)
            for i in range(self.img_ls_len)
        ]
        print(f"  Done.")

    def __getitem__(self, idx):
        return self._cache[idx]


print("Loading YOLOv8 detector (frozen)...")
detector = YOLOv8()
for p in detector.model.parameters():
    p.requires_grad = False

CFG = SimpleNamespace(
    canary_cls_id=CANARY_CLS_ID, canary_size=80,
    img_size=640, person_conf=0.05, overlap_thresh=0.4,
    defensive_patch_location='cc', eval_no_overlap=True,
    margin_size=0, faster=False,
)

print("Loading training data...")
train_dataset = InMemoryYolov8Dataset(
    os.path.join(DATA_DIR, 'benign'),
    os.path.join(DATA_DIR, 'adversarial'),
    os.path.join(DATA_DIR, 'benign_label'),
    CFG.img_size,
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, drop_last=True,
)
print(f"Dataloader ready: {len(train_dataset)} pairs, batch_size={BATCH_SIZE}")

In [ ]:
# =============================================
#  SEQUENTIAL TRAINING LOOP
# =============================================
# For multi-GPU HPC, prefer train_ensemble_parallel.py which spawns workers
# across all visible GPUs. This cell is the single-GPU fallback.

import torch.optim as optim

wall_start = time.time()
completed = 0
skipped = 0

for gen_idx in range(GEN_START, GEN_END):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'gen_{gen_idx:04d}.pt')

    if os.path.exists(ckpt_path):
        skipped += 1
        if skipped <= 3 or skipped % 100 == 0:
            print(f"  gen_{gen_idx:04d}.pt already exists, skipping")
        continue

    gen_start_time = time.time()
    seed = gen_idx + 1
    freeze_seed(seed)

    g = GenerativeCanary(z_dim=Z_DIM).to(device)
    optimizer = optim.Adam(g.parameters(), lr=LR, amsgrad=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(1, EPOCHS + 1):
        g.train()
        for benign_input, _, adv_input in train_loader:
            benign_input = benign_input.to(device)
            adv_input = adv_input.to(device)
            bs = benign_input.shape[0]

            eta = torch.randn(bs, Z_DIM, device=device)
            bbox = torch.rand(bs, 4, device=device)
            z = torch.cat([eta, bbox], dim=1)

            canary_patches = g(z)

            adv_pos_all, ben_pos_all = [], []
            detector.model.eval()
            for bi in range(bs):
                c_bi = canary_patches[bi]
                _, ap = add_defensivepatch_into_tensor(
                    detector, CFG, adv_input[bi:bi+1], c_bi, random_palce=True
                )
                _, bp = add_defensivepatch_into_tensor(
                    detector, CFG, benign_input[bi:bi+1], c_bi, random_palce=True
                )
                adv_pos_all.extend(ap)
                ben_pos_all.extend(bp)

            adv_batch = make_yolo_train_label(adv_input, adv_pos_all)
            ben_batch = make_yolo_train_label(benign_input, ben_pos_all)

            detector.model.train()
            adv_loss, _ = detector.model.model(adv_batch)
            ben_loss, _ = detector.model.model(ben_batch)

            loss = WEIGHT * ben_loss - adv_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        scheduler.step()

    # Save final checkpoint
    torch.save({
        'generator_state_dict': g.state_dict(),
        'seed': seed,
        'gen_idx': gen_idx,
        'epoch': EPOCHS,
    }, ckpt_path)

    elapsed = time.time() - gen_start_time
    completed += 1
    print(f"Generator {gen_idx}/{N_GENERATORS - 1} done "
          f"(seed={seed}) — {elapsed:.1f}s "
          f"[{completed} trained, {skipped} skipped]")

    if completed % 50 == 0:
        wall = time.time() - wall_start
        rate = completed / wall if wall > 0 else 0
        remaining = (GEN_END - GEN_START - completed - skipped) / rate if rate > 0 else 0
        print(f"  === {completed} done in {wall/60:.1f} min, "
              f"{rate*3600:.1f} gen/hr, ETA {remaining/3600:.1f}h ===")

total_wall = time.time() - wall_start
print(f"\nTraining complete. {completed} trained, {skipped} skipped, "
      f"{total_wall/60:.1f} min total.")

## Section 2: Local Inference and Evaluation

After training on HPC, download `ensemble_1000/checkpoints/` to this machine
and run the cells below on the RTX 3070 Ti.

In [ ]:
import glob
import torch
from generative_canary import GenerativeCanary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_paths = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'gen_*.pt')))
print(f"Found {len(ckpt_paths)} checkpoints in {CHECKPOINT_DIR}")

ensemble = []
ensemble_on_gpu = True
try:
    for i, cp in enumerate(ckpt_paths):
        g = GenerativeCanary(z_dim=Z_DIM).to(device)
        ckpt = torch.load(cp, map_location=device)
        g.load_state_dict(ckpt['generator_state_dict'])
        g.eval()
        ensemble.append(g)
        if (i + 1) % 100 == 0:
            print(f"  Loaded {i+1}/{len(ckpt_paths)}")
except torch.cuda.OutOfMemoryError:
    print("  GPU OOM -- falling back to CPU-resident ensemble")
    for g in ensemble:
        g.cpu()
    ensemble_on_gpu = False
    for cp in ckpt_paths[len(ensemble):]:
        g = GenerativeCanary(z_dim=Z_DIM)
        ckpt = torch.load(cp, map_location='cpu')
        g.load_state_dict(ckpt['generator_state_dict'])
        g.eval()
        ensemble.append(g)

print(f"Ensemble ready: {len(ensemble)} generators on "
      f"{'GPU' if ensemble_on_gpu else 'CPU'}")

In [ ]:
import random
import matplotlib.pyplot as plt

# Pick 10 random generators, sample 2 canaries from each = 20 total
rng = random.Random(42)
picked_indices = rng.sample(range(len(ensemble)), min(10, len(ensemble)))

fig, axes = plt.subplots(2, len(picked_indices),
                          figsize=(2.5 * len(picked_indices), 5))
if len(picked_indices) == 1:
    axes = [[axes[0]], [axes[1]]]

all_samples = []
for col, gen_idx in enumerate(picked_indices):
    g = ensemble[gen_idx]
    if not ensemble_on_gpu:
        g = g.to(device)
    for row in range(2):
        eta = torch.randn(1, Z_DIM, device=device)
        bbox = torch.rand(1, 4, device=device)
        z = torch.cat([eta, bbox], dim=1)
        with torch.no_grad():
            c = g(z)
        all_samples.append(c[0])
        img = c[0].cpu().permute(1, 2, 0).numpy()
        ax = axes[row][col] if len(picked_indices) > 1 else axes[row]
        ax.imshow(img)
        ax.set_title(f'Gen {gen_idx}')
        ax.axis('off')
    if not ensemble_on_gpu:
        g.cpu()

fig.suptitle(f'K={len(ensemble)} Ensemble: 10 Random Generators, 2 Samples Each',
             fontsize=13)
plt.tight_layout()
plt.show()

# Diversity metrics
stacked = torch.stack(all_samples)
flat = stacked.view(len(all_samples), -1)
dists = torch.cdist(flat, flat, p=2)
mask = torch.triu(torch.ones(len(all_samples), len(all_samples),
                              device=device), diagonal=1).bool()
mean_l2 = dists[mask].mean().item()

import torch.nn.functional as F
norm = F.normalize(flat, p=2, dim=1)
cos_sim = norm @ norm.t()
mean_cos = cos_sim[mask].mean().item()

print(f"Mean pairwise L2 distance:   {mean_l2:.2f}")
print(f"Mean pairwise cosine sim:    {mean_cos:.4f}")
print(f"5-generator baseline L2:      29.56")
print(f"Ratio vs 5-gen baseline:      {mean_l2 / 29.56:.2f}x")

In [ ]:
import cv2
import numpy as np
import pandas as pd
from types import SimpleNamespace
from tqdm import tqdm
from IPython.display import display

from YOLOv8_Combiner import Canary
from ObjectDetector.fjn_yolov8 import FJN_YOLOv8 as YOLOv8

TEST_ROOT = '../Data/testeval/VOC07_YOLOv8/test'
attacks = ['AdvPatch', 'UPC', 'TCEGA', 'Natural']

print("Loading YOLOv8 detector for evaluation...")
detector = YOLOv8()
detector.model.eval()

eval_cfg = SimpleNamespace(
    canary_cls_id=CANARY_CLS_ID, canary_size=80, img_size=640,
    person_conf=0.05, overlap_thresh=0.4, defensive_patch_location='cc',
    eval_no_overlap=True, margin_size=0, faster=False,
)
canary_eval = Canary(eval_cfg, detector)

eval_rng = random.Random(301)

def sample_from_ensemble():
    g = eval_rng.choice(ensemble)
    if not ensemble_on_gpu:
        g_gpu = g.to(device)
        eta = torch.randn(1, Z_DIM, device=device)
        bbox = torch.rand(1, 4, device=device)
        with torch.no_grad():
            patch = g_gpu(torch.cat([eta, bbox], dim=1))
        g.cpu()
    else:
        eta = torch.randn(1, Z_DIM, device=device)
        bbox = torch.rand(1, 4, device=device)
        with torch.no_grad():
            patch = g(torch.cat([eta, bbox], dim=1))
    return patch

results_kgen = {}
for attack in attacks:
    adv_dir = os.path.join(TEST_ROOT, attack, 'adversarial')
    ben_dir = os.path.join(TEST_ROOT, attack, 'benign')
    TP = FP = FN = TN = 0

    adv_files = sorted([f for f in os.listdir(adv_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    for fname in tqdm(adv_files, desc=f'{attack} adv', leave=False):
        patch = sample_from_ensemble()
        canary_np = (patch[0].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        canary_eval.eval_canary = cv2.cvtColor(canary_np, cv2.COLOR_RGB2BGR)
        canary_eval.canary_cls_id = CANARY_CLS_ID
        img = cv2.imread(os.path.join(adv_dir, fname), 1)
        if canary_eval.eval_single(img):
            TP += 1
        else:
            FN += 1

    ben_files = sorted([f for f in os.listdir(ben_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    for fname in tqdm(ben_files, desc=f'{attack} ben', leave=False):
        patch = sample_from_ensemble()
        canary_np = (patch[0].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        canary_eval.eval_canary = cv2.cvtColor(canary_np, cv2.COLOR_RGB2BGR)
        canary_eval.canary_cls_id = CANARY_CLS_ID
        img = cv2.imread(os.path.join(ben_dir, fname), 1)
        if canary_eval.eval_single(img):
            FP += 1
        else:
            TN += 1

    P = TP / (TP + FP) if (TP + FP) > 0 else 0
    R = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    results_kgen[attack] = {'F1': F1, 'FPR': FPR,
                            'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN}
    print(f"  {attack:10s}  F1={F1:.3f}  FPR={FPR:.3f}")

# ---- Styled comparison table ----
paper = {'AdvPatch': 0.974, 'UPC': 0.936, 'TCEGA': 0.807, 'Natural': 0.871}
fixed17 = {'AdvPatch': 0.935, 'UPC': 0.877, 'TCEGA': 0.896, 'Natural': 0.900}
ens5 = {'AdvPatch': 0.935, 'UPC': 0.908, 'TCEGA': 0.905, 'Natural': 0.929}

rows = []
for attack in attacks:
    r = results_kgen[attack]
    rows.append({
        'Attack': attack,
        'Paper F1': paper[attack],
        'Fixed17 F1': fixed17[attack],
        '5-gen F1': ens5[attack],
        f'{len(ensemble)}-gen F1': r['F1'],
        f'{len(ensemble)}-gen FPR': r['FPR'],
        'Delta vs Paper': r['F1'] - paper[attack],
    })
df = pd.DataFrame(rows).set_index('Attack')

def color_f1(v):
    if isinstance(v, float) and v >= 0.90:
        return 'background-color: #c6efce; color: #006100; font-weight: bold'
    elif isinstance(v, float) and v >= 0.85:
        return 'background-color: #ffeb9c; color: #9c5700'
    return ''

def color_fpr(v):
    if isinstance(v, float) and v <= 0.05:
        return 'background-color: #c6efce; color: #006100; font-weight: bold'
    elif isinstance(v, float) and v <= 0.10:
        return 'background-color: #ffeb9c; color: #9c5700'
    return ''

def color_delta(v):
    if isinstance(v, float) and v > 0:
        return 'background-color: #c6efce; color: #006100; font-weight: bold'
    elif isinstance(v, float) and v < 0:
        return 'background-color: #ffc7ce; color: #9c0006; font-weight: bold'
    return ''

fmt = {c: '{:.3f}' for c in df.columns if 'F1' in c or 'FPR' in c}
fmt['Delta vs Paper'] = '{:+.3f}'

styled = (df.style
    .format(fmt)
    .applymap(color_f1, subset=[c for c in df.columns if 'F1' in c])
    .applymap(color_fpr, subset=[c for c in df.columns if 'FPR' in c])
    .applymap(color_delta, subset=['Delta vs Paper'])
    .set_caption(f'Mode #1 Canary -- {len(ensemble)}-Generator Ensemble on YOLOv8 / VOC07')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'),
                                           ('font-weight', 'bold'),
                                           ('margin-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#2c3e50'),
                                      ('color', 'white'),
                                      ('padding', '8px 12px'),
                                      ('text-align', 'center')]},
        {'selector': 'td', 'props': [('padding', '6px 12px'),
                                      ('text-align', 'center')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f8f9fa')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'),
                                         ('margin', '10px 0'),
                                         ('font-family', 'Segoe UI, Arial, sans-serif')]},
        {'selector': 'th, td', 'props': [('border', '1px solid #dee2e6')]},
    ]))

display(styled)

## Attacker Cost Analysis

With K generators the adaptive attacker must either:
1. **Brute-force all K**: optimize one adversarial patch that defeats every generator simultaneously — cost is linear in K.
2. **Gamble**: pick one generator and hope the defender uses it — probability 1/K.

In [ ]:
# Attacker cost analysis for K-generator ensemble

K = len(ensemble)

# Single-term adaptive attack cost (from FFF paper, adjusted)
# Paper reports 37 days for 6 canaries on RTX 3090
# -> per-term cost = 37 / 6 = 6.17 days
per_term_days_3090 = 37 / 6

# GPU speedup factors (approximate, sustained training FLOPs)
speedups = {
    'RTX 3090':   1.0,
    'B100':       11.0,   # ~11x 3090 for FP16 training
    'B200':       20.0,   # ~20x 3090 for FP16 training
    '8x B200':    160.0,  # 8 GPUs, ~20x each
}

total_per_term_days = {name: per_term_days_3090 / s
                       for name, s in speedups.items()}
k_term_days = {name: v * K for name, v in total_per_term_days.items()}

print(f"=" * 72)
print(f"ATTACKER COST: defeating K={K} generators simultaneously")
print(f"=" * 72)
print(f"{'GPU':<12s} | {'Per-term cost':>15s} | {'K-term cost':>18s}")
print(f"-" * 72)
for name, days in k_term_days.items():
    per_term = total_per_term_days[name]
    if days > 365:
        days_str = f"{days/365:.1f} years"
    elif days > 30:
        days_str = f"{days:.0f} days ({days/30:.1f} months)"
    else:
        days_str = f"{days:.1f} days"
    if per_term > 1:
        per_term_str = f"{per_term:.2f} days"
    else:
        per_term_str = f"{per_term*24:.1f} hours"
    print(f"{name:<12s} | {per_term_str:>15s} | {days_str:>18s}")
print(f"=" * 72)
print()
print(f"Gambling attack (pick one generator, hope for match):")
print(f"  Success probability: 1/{K} = {100/K:.3f}%")
print()

# Defender cost
defender_min_per_gen = 5  # minutes per generator on T4
defender_sequential_min = K * defender_min_per_gen
defender_parallel_4gpu_hours = defender_sequential_min / 4 / 60

print(f"Defender cost:")
print(f"  Sequential on 1x T4:  {defender_sequential_min:.0f} min "
      f"({defender_sequential_min/60:.1f} hours)")
print(f"  Parallel on 4x T4:    {defender_parallel_4gpu_hours:.1f} hours")
print()

# Asymmetry
attacker_3090_days = k_term_days['RTX 3090']
defender_days = defender_parallel_4gpu_hours / 24
ratio = attacker_3090_days / defender_days if defender_days > 0 else float('inf')
print(f"Cost asymmetry (attacker RTX 3090 / defender 4x T4): ~{ratio:.0f}x")
print(f"\nConclusion: for K={K}, a well-funded attacker with 8x B200 still needs "
      f"{k_term_days['8x B200']:.0f} days to prepare a guaranteed attack, while "
      f"the defender reprepares the entire ensemble in ~{defender_parallel_4gpu_hours:.0f} hours.")